<a href="https://colab.research.google.com/github/AKookani/BrickwallCliffordCircuit/blob/main/Python_translation_of_NN_for_Measurement_Induced_Phase_Transitions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Binary Vector to Pauli String

In [23]:
!pip install stim

In [24]:
import numpy as np
import stim

def return_pauli(p, sign=0x0):
    # Converts a binary vector p into a PauliOperator:
    # first half = X components, second half = Z components, with optional phase sign.
    p = np.asarray(p, dtype=bool)
    n = len(p)
    half = n // 2
    x_part = p[:half]
    z_part = p[half:]

    # Build Pauli string: combine X and Z parts per qubit
    # X=1, Z=2, Y=3 (X+Z), I=0
    pauli_chars = []
    for x, z in zip(x_part, z_part):
        if x and z:
            pauli_chars.append('Y')
        elif x:
            pauli_chars.append('X')
        elif z:
            pauli_chars.append('Z')
        else:
            pauli_chars.append('I')

    # Map sign byte to stim's phase convention (0x0=+1, 0x1=+i, 0x2=-1, 0x3=-i)
    sign_map = {0x0: '+', 0x1: '+', 0x2: '-', 0x3: '-'}
    sign_char = sign_map.get(sign, '+')

    pauli_str = sign_char + ''.join(pauli_chars)
    return stim.PauliString(pauli_str)

## X part and Z part — The Symplectic Representation

In quantum computing, every Pauli operator is built from these 4 single-qubit matrices:

$$I = \begin{pmatrix}1&0\\0&1\end{pmatrix}, \quad X = \begin{pmatrix}0&1\\1&0\end{pmatrix}, \quad Z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}, \quad Y = iXZ = \begin{pmatrix}0&-i\\i&0\end{pmatrix}$$

Instead of storing these matrices directly, we store them as **two bits per qubit**:

```
Does this qubit have an X?  → x bit (0 or 1)
Does this qubit have a Z?   → z bit (0 or 1)
```

That's it. Y is just "has both X and Z".

---

### Concrete Example

Say you have a 3-qubit system with operator **X ⊗ Y ⊗ Z** (⊗ means "applied to each qubit"):

| Qubit | Pauli | Has X? | Has Z? |
|-------|-------|--------|--------|
| 0 | X | 1 | 0 |
| 1 | Y | 1 | 1 |
| 2 | Z | 0 | 1 |

So the flat binary vector `p` is:

```
p = [ 1, 1, 0,   0, 1, 1 ]
      ↑  ↑  ↑    ↑  ↑  ↑
      X-part     Z-part
    (q0,q1,q2) (q0,q1,q2)
```
---

### Why this representation?
This is called the **symplectic representation** and it's used heavily in **stabilizer/Clifford** circuit simulation (which is what `QuantumClifford.jl` and `stim` are built for) because:
- It's **memory efficient** — just bits, no complex matrices
- Pauli multiplication becomes simple **bitwise XOR**
- It scales to thousands of qubits easily

## The Sign (Phase) of a Pauli Operator

### Why does a Pauli operator need a sign?

When you multiply Pauli matrices together, you don't always get a clean result — you pick up **phase factors**. For example:

$$X \cdot Z = -iY \qquad Z \cdot X = +iY$$

So the same set of X/Z bits can represent **4 different operators** depending on the phase. That's why the sign is stored separately.

### Concrete Example

Take the X/Z bits for a single qubit Y operator (`x=1, z=1`). Depending on the sign, you get 4 different operators:

| `sign` | Full operator |
|--------|--------------|
| `0x0` | +Y |
| `0x1` | +iY |
| `0x2` | −Y |
| `0x3` | −iY |

---

### When does the sign actually matter?

**1. Stabilizer states** — In stabilizer formalism, qubits are defined by their stabilizers. The sign tells you *which eigenstate* you're in:

```
+Z  stabilizes  |0⟩   (eigenvalue +1)
-Z  stabilizes  |1⟩   (eigenvalue -1)
```
So `+Z` and `−Z` describe **completely different physical states**, even though the X/Z bits are identical.

**2. Pauli multiplication** — When you multiply two Pauli operators, the signs multiply too (and can accumulate factors of ±i). Tracking this correctly is essential for simulation.

**3. Measurement outcomes** — When you measure a Pauli operator, the sign flips tells you whether the result is +1 or −1, which corresponds to the **physical 0 or 1 outcome** of the measurement.


In [25]:
p = [1, 0, 1, 0, 1, 0]  # n=3 qubits: X=[1,0,1], Z=[0,1,0]
return_pauli(p, sign=0x0)
# → +XZX   (stim.PauliString)

stim.PauliString("+XZX")

# 2. Load and Format 2-Qubit Clifford Unitaries (→ Symplectic Tableau)

In [26]:
import requests

def RandUnitaries():
    # Reads 2-qubit Clifford unitaries from a text file and reorders them into
    # a symplectic tableau format: each group of 4 rows is rearranged as
    # [Z1, X1, X2, Z2], then columns 1 and 4 are swapped.

    Nline = 2880
    unitaries = np.zeros((Nline, 4), dtype=int)

    # Download the file from GitHub
    url = "https://raw.githubusercontent.com/AKookani/BrickwallCliffordCircuit/main/twoUnitaries.txt"
    response = requests.get(url)
    response.raise_for_status()  # Check for download errors

    # Read the content line by line
    lines = response.text.strip().split('\n')

    for line_idx, s in enumerate(lines):
        s = s.strip()
        unitaries[line_idx, 0] = int(s[0])
        unitaries[line_idx, 1] = int(s[2])
        unitaries[line_idx, 2] = int(s[4])
        unitaries[line_idx, 3] = int(s[6])

    randU = np.zeros((Nline, 4), dtype=int)

    for n in range(Nline // 4):
        block = unitaries[4*n : 4*n+4, :]         # shape (4, 4)
        randU[4*n + 0, :] = block[:, 2]  # Z1 row
        randU[4*n + 1, :] = block[:, 0]  # X1 row
        randU[4*n + 3, :] = block[:, 3]  # Z2 row
        randU[4*n + 2, :] = block[:, 1]  # X2 row

    # Swap columns 0 and 3
    randU[:, [0, 3]] = randU[:, [3, 0]]

    return randU

It parses `twoUnitaries.txt`[Link to the .txt file](https://github.com/AKookani/BrickwallCliffordCircuit/blob/main/twoUnitaries.txt), which stores 2-qubit Clifford unitaries as rows of 4 integers (space-separated characters). The 2880 rows are grouped in blocks of 4, each representing one unitary's symplectic **tableau (A binary table where rows = stabilizer generators, columns = qubits)**. The rows within each block are reordered from the file's column order `[col1, col2, col3, col4]` into the row arrangement `[Z1, X1, X2, Z2]`. Finally, the first and last columns of the entire matrix are swapped, likely to match a specific symplectic convention used downstream.

## `RandUnitaries()` — Overview

The function loads and reformats a table of 2-qubit Clifford unitaries for use in a symplectic (Pauli) tableau framework.

**1. Load raw data**
Reads 2880 lines from `twoUnitaries.txt`, parsing 4 integer values per line (space-separated characters) into a matrix. Each row represents one element of a unitary's symplectic representation.

**2. Reorder into symplectic tableau convention**
The data arrives in groups of 4 rows per unitary. Each group is reordered so the rows follow the convention `[Z1, X1, X2, Z2]` — matching the standard ordering of Pauli generators in a symplectic tableau (X and Z stabilizers for each qubit).

**3. Swap columns 0 and 3**
Swaps the first and last columns across all rows, likely reconciling a difference between the file's qubit ordering and the expected internal convention (e.g., qubit 1 ↔ qubit 2, or bit ordering within the symplectic vector).

**Net result:** Returns `randU`, a reformatted matrix of 2-qubit Clifford unitaries ready for use in stabilizer/symplectic circuit simulations (e.g., randomized benchmarking or random Clifford circuit sampling).

In [27]:
result = RandUnitaries()
print(f"Shape of result: {result.shape}")
print(f"Data type: {result.dtype}")
print("\nFirst 10 rows:")
print(result[:10])
print("\nLast 10 rows:")
print(result[-10:])
print(f"\nUnique values: {np.unique(result)}")

Shape of result: (2880, 4)
Data type: int64

First 10 rows:
[[0 0 0 1]
 [0 0 1 0]
 [1 0 0 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]
 [1 1 0 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]]

Last 10 rows:
[[0 1 1 1]
 [1 1 1 1]
 [0 0 1 1]
 [0 1 1 0]
 [1 1 1 1]
 [1 0 1 1]
 [0 0 1 1]
 [0 1 1 0]
 [1 0 1 1]
 [1 1 1 1]]

Unique values: [0 1]


# 3. Initialize Product State Stabilizer Tableau

In [28]:
def init_prod_state(N: int, ancilla: bool) -> np.ndarray:
    N = int(N)
    G = np.zeros((2 * N, 2 * N + 1), dtype=np.int8)

    for i in range(N):
        G[i,     2 * i]     = 1  # Z stabilizers: Z_i  → Z-block column
        G[i + N, 2 * i + 1] = 1  # X stabilizers: X_i  → X-block column

    return G

## What is this code doing?

Imagine you have **N qubits**. Each qubit has two "handles" you can grab:

- A **Z handle** (measures if the qubit is 0 or 1)
- An **X handle** (measures if the qubit is + or -)

This function just creates a **checklist** (a grid of 0s and 1s) that says:

> *"Qubit 1 has its Z handle here, its X handle there. Qubit 2 has its Z handle here, its X handle there. And so on..."*

---

## A concrete example — N=2 qubits

```
         q1_Z  q1_X  q2_Z  q2_X  | phase
Z of q1: [ 1     0     0     0   |   0  ]
Z of q2: [ 0     0     1     0   |   0  ]
X of q1: [ 0     1     0     0   |   0  ]
X of q2: [ 0     0     0     1   |   0  ]
```

Each row says **"I am watching this qubit, through this handle"**. Nothing more.

---

## The `ancilla` flag?

It does **nothing different** in this code — both branches produce the exact same grid. It's likely a placeholder for future changes.

---

## One-liner summary

> **It creates a map of where each qubit's Z and X controls are located, so a quantum simulator knows how to track them.**

In [29]:
for ancilla in [False, True]:
    for N in [1, 2, 3]:
        G = init_prod_state(N, ancilla)
        print(f"N={N}, ancilla={ancilla} → shape={G.shape}")
        print(G, "\n")

# Specific checks for N=2
G = init_prod_state(2, False)
assert G.shape == (4, 5),           "Shape should be (2N, 2N+1)"
assert G[0, 0] == 1,                "Z stabilizer qubit 0 → col 0"
assert G[1, 2] == 1,                "Z stabilizer qubit 1 → col 2"
assert G[2, 1] == 1,                "X stabilizer qubit 0 → col 1"
assert G[3, 3] == 1,                "X stabilizer qubit 1 → col 3"
assert G[:, 4].sum() == 0,          "Phase column should be all zeros"
assert G.sum() == 2 * 2,            "Exactly 2N ones in the tableau"
print("All assertions passed ✓")

N=1, ancilla=False → shape=(2, 3)
[[1 0 0]
 [0 1 0]] 

N=2, ancilla=False → shape=(4, 5)
[[1 0 0 0 0]
 [0 0 1 0 0]
 [0 1 0 0 0]
 [0 0 0 1 0]] 

N=3, ancilla=False → shape=(6, 7)
[[1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0]] 

N=1, ancilla=True → shape=(2, 3)
[[1 0 0]
 [0 1 0]] 

N=2, ancilla=True → shape=(4, 5)
[[1 0 0 0 0]
 [0 0 1 0 0]
 [0 1 0 0 0]
 [0 0 0 1 0]] 

N=3, ancilla=True → shape=(6, 7)
[[1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0]] 

All assertions passed ✓


# 4. Initialize Z-Only Stabilizer Tableau (Destabilizers = 0)

In [30]:
def init_prod_state_short(N: int, ancilla: bool) -> np.ndarray:
    # Initializes a stabilizer tableau of shape (2N, 2N+1) for N qubits.
    # Rows 0..N-1 are Z stabilizers (Z-block diagonal set to 1).
    # Rows N..2N-1 are zero (X destabilizers, unpopulated here).
    # The last column is reserved for the sign bit (unused/zero here).
    N = int(N)
    Ginit = np.zeros((2 * N, 2 * N + 1), dtype=np.int8)
    for i in range(N):
        Ginit[i, 2 * i] = 1  # Z stabilizer for qubit i (0-indexed)
    return Ginit

The function initializes a **stabilizer tableau** for a product state of N qubits (with or without ancilla). In both branches it does the same thing: creates a `(2N) × (2N+1)` matrix of zeros and sets Z stabilizers along the diagonal (in the Z-block). The `ancilla` flag doesn't actually change the output — both branches are identical.

# 5. Conditional Spin Mapping: f(a,b,c,d) → [-1,0,1] Values

In [31]:
def four_bit_func(a, b, c, d):
    """
    Selector function driven by control bits a, b.
    Maps binary inputs to ±1 spin-like values via (1-2x) transformation,
    common in Pauli/stabilizer or parity-based quantum computations.
    """
    a, b, c, d = np.asarray(a), np.asarray(b), np.asarray(c), np.asarray(d)

    return np.select(
        condlist=[
            (a == 0) & (b == 0),
            (a == 1) & (b == 1),
            (a == 1) & (b == 0),
            (a == 0) & (b == 1),
        ],
        choicelist=[
            np.zeros_like(c, dtype=float),
            c - d,
            d * (1 - 2 * c),
            c * (2 * d - 1),
        ],
        default=np.nan  # undefined for any input outside the four cases
    )

The function computes a value based on two control bits (`a`, `b`) and two data bits (`c`, `d`):

- `a=0, b=0` → returns 0
- `a=1, b=1` → returns `c - d`
- `a=1, b=0` → returns `d(1 - 2c)`
- `a=0, b=1` → returns `c(2d - 1)`

It looks like a **quantum-style parity/selector function** — the `(1-2x)` pattern maps bits `{0,1}` to `{+1,-1}`, which is classic in Pauli/stabilizer contexts.

# 6. Element-wise XOR (Modulo-2 Addition)

In [32]:
# Element-wise XOR of two vectors (equivalent to modulo-2 addition)
def xor(v1, v2):
    return (np.array(v1) + np.array(v2)) % 2

This function performs **element-wise modulo-2 addition** (logical XOR) on two vectors — the standard binary addition used in coding theory and quantum error correction.

NumPy handles the zipping and integer conversion implicitly, making it a clean one-liner.

# 7. Pauli Product in Binary Symplectic Form (with Phase Tracking)

In [36]:
def pauli_prod(P1, P2):
    # Product of two vectorial Pauli operators [x1,z1,...,xN,zN, phase].
    # Returns (new Pauli vector, phase coefficient 0 or 1).
    P1, P2 = np.asarray(P1), np.asarray(P2)
    N = round((len(P1) - 1) / 2)

    r1 = P1[2 * N]      # phase bit of P1 (0-indexed)
    r2 = P2[2 * N]      # phase bit of P2

    pauli_part = xor(P1[:2 * N], P2[:2 * N])   # XOR of Pauli bits (excludes phase)

    sumG = sum(
        four_bit_func(P1[2*i], P1[2*i + 1], P2[2*i], P2[2*i + 1])
        for i in range(N)
    )

    phase = abs(((2 * r1 + 2 * r2 + sumG) % 4) / 2)

    return pauli_part, phase

The function computes the **product of two Pauli operators** in the symplectic (binary vector) representation.

**Representation assumed:**
Each Pauli operator is stored as a vector of the form `[x₁, z₁, x₂, z₂, ..., xₙ, zₙ, r]` where the `xᵢ, zᵢ` pairs encode the single-qubit Pauli on qubit `i` (I, X, Y, or Z), and `r` is a phase bit.

**What it returns:**
- **`pauli_part`** — the Pauli vector of the product, computed by XOR-ing the two Pauli bit strings (this is the standard symplectic addition).
- **`phase`** — the resulting phase factor (0 or 1), which tracks the ±1 or ±i coefficient that arises when multiplying non-commuting Paulis (e.g. XY = iZ).

**Why `four_bit_func` is needed:**
When you multiply two single-qubit Paulis, you get a phase contribution that depends on both operators. The `four_bit_func` captures this per-qubit phase accumulation by looking at the `(xᵢ, zᵢ)` pairs of both operators. The total `sumG` accumulates these contributions across all N qubits, and the final `% 4 / 2` collapses it into a binary phase (0 or 1).

**In short:** it implements the **symplectic group law for the N-qubit Pauli group**, which is the core operation underlying stabilizer/Clifford circuit simulation — exactly the kind of thing used in surface code or stabilizer formalism work.

In [39]:
P1 = [1, 0, 1, 0, 0]  # X on qubit 1, X on qubit 2, phase 0
P2 = [0, 1, 0, 1, 0]  # Z on qubit 1, Z on qubit 2, phase 0

sum_pauli, phase = pauli_prod(P1, P2)
print("sum_pauli:", sum_pauli)
print("phase:", phase)

sum_pauli: [1 1 1 1]
phase: 1.0


In [40]:
# === Test cases ===
# Format: [x1, z1, x2, z2, ..., phase]  for N qubits

# 1-qubit Paulis: X=[1,0,0], Y=[1,1,0], Z=[0,1,0], I=[0,0,0]
X = [1, 0, 0]
Y = [1, 1, 0]
Z = [0, 1, 0]
I = [0, 0, 0]

tests_1q = [
    ("X * X", X, X),   # = I, phase 0
    ("X * Y", X, Y),   # = iZ, phase 1
    ("Y * X", Y, X),   # = -iZ, phase 1
    ("X * Z", X, Z),   # = -iY, phase 1
    ("Z * X", Z, X),   # = iY, phase 1
    ("Y * Z", Y, Z),   # = iX, phase 1
    ("Z * Y", Z, Y),   # = -iX, phase 1
    ("X * I", X, I),   # = X, phase 0
    ("I * I", I, I),   # = I, phase 0
]

print("=== 1-qubit Pauli products ===")
print(f"{'Test':<12} {'Symplectic':<15} {'Phase'}")
print("-" * 35)
for label, P1, P2 in tests_1q:
    symp, phase = pauli_prod(P1, P2)
    print(f"{label:<12} {str(symp):<15} {phase}")

# 2-qubit Paulis: [x1,z1, x2,z2, phase]
XX = [1, 0, 1, 0, 0]
ZZ = [0, 1, 0, 1, 0]
YY = [1, 1, 1, 1, 0]
IX = [0, 0, 1, 0, 0]
XI = [1, 0, 0, 0, 0]

tests_2q = [
    ("XX * ZZ", XX, ZZ),   # = YY up to phase
    ("ZZ * XX", ZZ, XX),
    ("YY * XX", YY, XX),
    ("IX * XI", IX, XI),   # commute, phase 0
]

print("\n=== 2-qubit Pauli products ===")
print(f"{'Test':<12} {'Symplectic':<20} {'Phase'}")
print("-" * 40)
for label, P1, P2 in tests_2q:
    symp, phase = pauli_prod(P1, P2)
    print(f"{label:<12} {str(symp):<20} {phase}")

=== 1-qubit Pauli products ===
Test         Symplectic      Phase
-----------------------------------
X * X        [0 0]           0.0
X * Y        [0 1]           1.5
Y * X        [0 1]           0.5
X * Z        [1 1]           0.5
Z * X        [1 1]           1.5
Y * Z        [1 0]           1.5
Z * Y        [1 0]           0.5
X * I        [1 0]           0.0
I * I        [0 0]           0.0

=== 2-qubit Pauli products ===
Test         Symplectic           Phase
----------------------------------------
XX * ZZ      [1 1 1 1]            1.0
ZZ * XX      [1 1 1 1]            1.0
YY * XX      [0 1 0 1]            1.0
IX * XI      [1 0 1 0]            0.0


# 8.